# m04 · Nhiên liệu và quá trình cháy: các phép tính kiểm chứng

Sổ tay này tính lại bằng Python các ví dụ chính của module m04 (Slide Ch.4; Cengel Ch.15; Farokhi Ch.7; El-Sayed Ch.10). Mỗi mục có ba phần: câu hỏi mà phương pháp trả lời → mã → diễn giải số in thật. Chạy từ trên xuống. Các bảng A-4, A-5, A-26, A-27 và các cột h, s° của A-18…A-23 (đọc từ ảnh trang phụ lục) được nhúng ở ô mã đầu; đường độc lập dùng sympy, scipy và iapws. Phần định luật 1 và nhiệt độ ngọn lửa đoạn nhiệt là Bổ sung từ sách (không có trên slide).

## 1. Cân bằng phương trình cháy bằng ma trận nguyên tố

**Câu hỏi:** bốn cân bằng nguyên tố (C, H, O, N) cho nghiệm gì với octan cháy trong không khí chứa 20 kmol O₂ (slide tr.13–17), và AF khác nhau bao nhiêu khi M_air = 29 hay 28,97?

In [1]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
x, y, z, w = sp.symbols('x y z w')
sol = sp.solve([sp.Eq(8, x), sp.Eq(18, 2 * y), sp.Eq(20 * 2, 2 * x + y + 2 * z), sp.Eq(20 * sp.Rational(376, 100), w)], [x, y, z, w], dict=True)[0]
print('x, y, z, w =', sol[x], sol[y], sol[z], sol[w], '| moles in:', 1 + 20 * sp.Rational(476, 100), '| moles out:', sum(sol.values()))
M_oct = A27['C8H18']['M']
AF29 = 20 * 4.76 * M_AIR_ROUND / (8 * 12 + 9 * 2); AFx = 20 * 4.76 * M_AIR_EXACT / M_oct
print(f"AF slide (29, 114) = {AF29:.2f} kg/kg; AF with 28.97 and {M_oct} = {AFx:.2f} kg/kg; difference {100 * (AF29 / AFx - 1):.2f} %")

x, y, z, w = 8 9 15/2 376/5 | moles in: 481/5 | moles out: 997/10
AF slide (29, 114) = 24.22 kg/kg; AF with 28.97 and 114.231 = 24.14 kg/kg; difference 0.31 %


**Đọc kết quả:** nghiệm (8; 9; 15/2; 376/5) = (8; 9; 7,5; 75,2) khớp slide tr.15–16. Số mol vào 96,2, ra 99,7: số mol không bảo toàn (tr.11). AF = 24,22 (slide 24,2) với 29 và 114; với 28,97 và 114,231 là 24,14: slide làm tròn M_air, lệch 0,31 %.

## 2. AF theo lượng không khí cho ba nhiên liệu

**Câu hỏi:** AF thay đổi thế nào theo % không khí lý thuyết cho CH₄, C₃H₈, C₈H₁₈ (slide tr.21–25)? Dùng phép cân bằng sympy và so với công thức a_th = c + h/4.

In [2]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
M = dict(CH4=16.0, C3H8=44.0, C8H18=114.0)        # rounded masses (C = 12, H = 1) as on the slides
for fu in ('CH4', 'C3H8', 'C8H18'):
    e = comp_el(fu); line = []
    for lam in (1.0, 1.2, 1.5, 2.0):
        b = balance(fu, lam); af = b['air'] * M_AIR_ROUND / M[fu]; line.append(f"{lam * 100:.0f} %: AF {af:.2f}, O2 left {b['O2']:.2f}")
    print(fu, f"a_th = {balance(fu, 1.0)['ath']:.2f} (c + h/4 = {e['C'] + e['H'] / 4:.2f}) |", ' | '.join(line))

CH4 a_th = 2.00 (c + h/4 = 2.00) | 100 %: AF 17.25, O2 left 0.00 | 120 %: AF 20.71, O2 left 0.40 | 150 %: AF 25.88, O2 left 1.00 | 200 %: AF 34.51, O2 left 2.00
C3H8 a_th = 5.00 (c + h/4 = 5.00) | 100 %: AF 15.69, O2 left 0.00 | 120 %: AF 18.82, O2 left 1.00 | 150 %: AF 23.53, O2 left 2.50 | 200 %: AF 31.37, O2 left 5.00
C8H18 a_th = 12.50 (c + h/4 = 12.50) | 100 %: AF 15.14, O2 left 0.00 | 120 %: AF 18.16, O2 left 2.50 | 150 %: AF 22.70, O2 left 6.25 | 200 %: AF 30.27, O2 left 12.50


**Đọc kết quả:** a_th đúng bằng c + h/4 (2; 5; 12,5); AF = (% lý thuyết/100) × AF hợp thức: 17,25; 15,69; 15,14 kg/kg ở 100 %, tăng tuyến tính (hình trong module). O₂ dư bằng (λ − 1) a_th.

## 3. Điểm sương: nội suy A-5 và IAPWS (slide tr.27–32)

**Câu hỏi:** điểm sương của sản phẩm etan 20 % thừa ở 100 kPa là bao nhiêu, và nội suy tuyến tính bảng A-5 lệch bao nhiêu so với phương trình trạng thái?

In [3]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
b = balance('C2H6', 1.2); n_prod = b['CO2'] + b['H2O'] + b['O2'] + b['N2']
Pv = b['H2O'] / n_prod * 100.0
print(f"N_prod = {n_prod:.2f} kmol; P_v = {Pv:.2f} kPa; T_dp (A-5 interpolation) = {Tsat_A5(Pv):.2f} C; T_dp (IAPWS) = {Tsat_iapws(Pv):.2f} C; slide 52.3")
err = max(abs(Tsat_A5(P) - Tsat_iapws(P)) for P in np.arange(7.5, 200.0, 0.25))
print(f"largest |A-5 interpolation - IAPWS| for P = 7.5...200 kPa: {err:.3f} C")

N_prod = 21.49 kmol; P_v = 13.96 kPa; T_dp (A-5 interpolation) = 52.27 C; T_dp (IAPWS) = 52.49 C; slide 52.3
largest |A-5 interpolation - IAPWS| for P = 7.5...200 kPa: 0.438 C


**Đọc kết quả:** N_prod = 21,49; P_v = 13,96 kPa; nội suy A-5 cho 52,27 °C (slide làm tròn 52,3), IAPWS cho 52,49 °C: lệch 0,22 °C là sai số nội suy tuyến tính trên hai hàng cách nhau 5 kPa. Trên 7,5–200 kPa lệch lớn nhất 0,438 °C, vì vậy các câu hỏi điểm sương khai dung sai 0,5 °C.

## 4. Phân tích cháy ngược từ thành phần khô (slide tr.33–39)

**Câu hỏi:** từ CO₂ 10,02 %, CO 0,88 %, O₂ 5,62 %, N₂ 83,48 % (khô) của octan, AF, phần trăm không khí lý thuyết và lượng nước ngưng ở 25 °C, 100 kPa là bao nhiêu? Slide làm tròn x = 1,36: ảnh hưởng gì?

In [4]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
R = lambda v: sp.nsimplify(str(v), rational=True)
CO2, CO, O2, N2 = R(10.02), R(0.88), R(5.62), R(83.48)
x, a, b = sp.symbols('x a b')
s = sp.solve([sp.Eq(a * sp.Rational(376, 100), N2), sp.Eq(8 * x, CO2 + CO), sp.Eq(18 * x, 2 * b)], [x, a, b], dict=True)[0]
print(f"a = {float(s[a]):.4f}, x = {float(s[x]):.4f}, b = {float(s[b]):.4f}; O2 check: a - (CO2 + CO/2 + O2 + b/2) = {float(s[a] - (CO2 + CO/2 + O2 + s[b]/2)):.4f}")
air = float(s[a] / s[x]); AF = air * 4.76 * 29 / 114; pct = air / 12.5 * 100
print(f"O2 per kmol fuel = {air:.3f}; AF = {AF:.2f} kg/kg (slide 19.76); % theoretical = {pct:.1f} % (slide 131 %)")
Nv = float(s[b] / s[x]); Ntot = float((CO2 + CO + O2 + N2 + s[b]) / s[x]); y = IAPWS97(T=298.15, x=0).P * 1000 / 100.0
Nw = brentq(lambda w: (Nv - w) / (Ntot - w) - y, 0, Nv - 1e-9)
print(f"products {Ntot:.2f} kmol (slide 82.53); N_w = {Nw:.2f} kmol = {Nw / Nv * 100:.1f} % of the vapor (slide 6.59 kmol, 73 %)")

a = 22.2021, x = 1.3625, b = 12.2625; O2 check: a - (CO2 + CO/2 + O2 + b/2) = -0.0091
O2 per kmol fuel = 16.295; AF = 19.73 kg/kg (slide 19.76); % theoretical = 130.4 % (slide 131 %)
products 82.39 kmol (slide 82.53); N_w = 6.60 kmol = 73.3 % of the vapor (slide 6.59 kmol, 73 %)


**Đọc kết quả:** a = 22,2021; x = 1,3625 (slide 1,36); b = 12,2625 (slide 12,24). Kiểm O₂ lệch 0,0091 do làm tròn thành phần. AF = 19,73 (slide 19,76), 130,4 % lý thuyết (slide 131 %: 16,32/12,5 = 130,56 theo số của slide). N_w = 6,60 kmol (73,3 %): khớp 6,59 kmol và 73 % của slide, giải bằng brentq với P_sat của IAPWS.

## 5. Entanpi cháy từ entanpi hình thành, đối chiếu nhiệt trị A-27 (slide tr.53–54)

**Câu hỏi:** h̄_C của octan lỏng (nước lỏng) là bao nhiêu, nhiệt trị cao/thấp theo kg là bao nhiêu và HHV − LHV có đúng bằng m·h_fg (h_fg từ IAPWS) không?

In [5]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
hf = F26
hC_l = 8 * hf['CO2(g)'] + 9 * hf['H2O(ᐉ)'] - hf['C8H18(ᐉ)']; hC_g = 8 * hf['CO2(g)'] + 9 * hf['H2O(g)'] - hf['C8H18(ᐉ)']
M = A27['C8H18']['M']
print(f"h_C (liquid H2O) = {hC_l:,.0f} kJ/kmol = {hC_l / M:,.0f} kJ/kg; A-27 HHV = {A27['C8H18']['HHV']:,.0f} kJ/kg")
print(f"h_C (H2O vapor) = {hC_g:,.0f} kJ/kmol = {hC_g / M:,.0f} kJ/kg; A-27 LHV = {A27['C8H18']['LHV']:,.0f} kJ/kg")
hfg = (IAPWS97(T=298.15, x=1).h - IAPWS97(T=298.15, x=0).h)
m_H2O = 9 * 18.015 / M
print(f"m h_fg = {m_H2O:.3f} kg/kg x {hfg:.1f} kJ/kg = {m_H2O * hfg:,.0f} kJ/kg; HHV - LHV (A-26) = {(hC_g - hC_l) / M:,.0f} kJ/kg; A-27: {A27['C8H18']['HHV'] - A27['C8H18']['LHV']:,.0f}")

h_C (liquid H2O) = -5,470,680 kJ/kmol = -47,891 kJ/kg; A-27 HHV = 47,890 kJ/kg
h_C (H2O vapor) = -5,074,590 kJ/kmol = -44,424 kJ/kg; A-27 LHV = 44,430 kJ/kg
m h_fg = 1.419 kg/kg x 2441.7 kJ/kg = 3,466 kJ/kg; HHV - LHV (A-26) = 3,467 kJ/kg; A-27: 3,460


**Đọc kết quả:** h̄_C = -5,470,680 kJ/kmol (slide làm tròn −5 471 000) = -47,891 kJ/kg; HHV A-27 = 47 890 (lệch 0,002 %). Với nước hơi: -44,424 kJ/kg (A-27 44 430). m·h_fg = 1,419 × 2441,7 = 3,466 kJ/kg, đúng bằng HHV − LHV của A-26 (3,467).

## 6. Entanpi hiển nhiệt h̄(T) − h̄(298 K): bảng, tích phân T ds° và đa thức cp

**Câu hỏi:** ví dụ tr.44 (N₂ ở 500 K: 14 581 − 8669 = 5912 kJ/kmol) có khớp tích phân T ds° trên chính các cột s° của bảng và đa thức cp của A-2c không?

In [6]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
print(f"N2 500 K: table {sens_table('N2', 500.0):.1f}; integral of T ds0 {sens_Tds('N2', 500.0):.1f}; A-2c polynomial {sens_poly('N2', 500.0):.1f} kJ/kmol")
for g, T in (('CO2', 1000.0), ('H2O', 1000.0), ('O2', 1500.0), ('N2', 1500.0)):
    t, i, p = sens_table(g, T), sens_Tds(g, T), sens_poly(g, T)
    print(f"{g:3s} {T:.0f} K: table {t:9.0f}; T ds0 {i:9.0f} ({100 * (i / t - 1):+.2f} %); polynomial {p:9.0f} ({100 * (p / t - 1):+.2f} %)")

N2 500 K: table 5912.0; integral of T ds0 5910.2; A-2c polynomial 5937.4 kJ/kmol
CO2 1000 K: table     33405; T ds0     33408 (+0.01 %); polynomial     33367 (-0.11 %)
H2O 1000 K: table     25978; T ds0     25982 (+0.02 %); polynomial     26041 (+0.24 %)
O2  1500 K: table     40610; T ds0     40606 (-0.01 %); polynomial     40724 (+0.28 %)
N2  1500 K: table     38404; T ds0     38402 (-0.01 %); polynomial     38430 (+0.07 %)


**Đọc kết quả:** N₂ ở 500 K: bảng 5912,0 (slide 5912), tích phân T ds° cho giá trị gần như trùng (hai cột h và s° của bảng nhất quán), đa thức cp lệch dưới 1 % (A-2c tự khai sai số lớn nhất 0,6–1,2 %). Vì vậy đường chính dùng cột h; hai đường kia là kiểm độc lập.

## 7. Định luật 1 cho buồng đốt: Cengel 15–55 (bổ sung §15-4)

**Câu hỏi:** benzen khí cháy với 95 % không khí lý thuyết, sản phẩm ra ở 1000 K (một phần C thành CO): nhiệt truyền ra khỏi buồng đốt là bao nhiêu và có khớp 2 112 800 kJ/kmol in của sách không?

In [7]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
hf = F26; n2 = 0.95 * 7.5 * 3.76
# C6H6 + 7.125 (O2 + 3.76 N2) -> x CO2 + y CO + 3 H2O + n2 N2 ; O: 14.25 = 2x + y + 3, x + y = 6
x, y = sp.symbols('x y'); s = sp.solve([sp.Eq(x + y, 6), sp.Eq(2 * x + y + 3, sp.Rational(1425, 100))], [x, y], dict=True)[0]
xc, yc = float(s[x]), float(s[y]); print('x (CO2), y (CO) =', xc, yc, f'; y_CO = {yc / (xc + yc + 3 + n2) * 100:.2f} %')
prod = [('CO2', xc, 'CO2(g)'), ('CO', yc, 'CO(g)'), ('H2O', 3.0, 'H2O(g)'), ('N2', n2, 'N2(g)')]
for tag, sens in (('table h column', sens_table), ('integral of T ds0', sens_Tds)):
    Hp = sum(n * (hf[k] + sens(g, 1000.0)) for g, n, k in prod); Q = hf['C6H6(g)'] - Hp
    print(f"{tag:18s}: H_prod = {Hp:,.0f}; Q_out = {Q:,.0f} kJ/kmol  (printed 2,112,800; difference {100 * (Q / 2112800 - 1):+.4f} %)")

x (CO2), y (CO) = 5.25 0.75 ; y_CO = 2.10 %
table h column    : H_prod = -2,029,849; Q_out = 2,112,779 kJ/kmol  (printed 2,112,800; difference -0.0010 %)
integral of T ds0 : H_prod = -2,029,871; Q_out = 2,112,801 kJ/kmol  (printed 2,112,800; difference +0.0000 %)


**Đọc kết quả:** x = 5,25; y = 0,75 ⇒ y_CO = 2,10 % (in 2,1 %). Q_ra = 2,112,779 kJ/kmol bằng cột h của bảng, chênh 0,0010 % so với 2 112 800 in; tích phân T ds° (cột s°) cho kết quả trùng trong 0,01 %. Đã thử cùng đường giải trên 15–56, 15–60, 15–67, 15–70 trước khi dùng cho 15–53 (không có đáp án in).

## 8. Nhiệt độ ngọn lửa đoạn nhiệt: Cengel 15–70, Ví dụ 15–10 và Farokhi Ví dụ 7.5 (bổ sung)

**Câu hỏi:** giải H_prod(T) = H_react bằng brentq với h̄(T) từ bảng (nội suy) và từ tích phân T ds°: kết quả có khớp 2251,4 K (15–70) và 1789 K (Ví dụ 15–10) in của sách không? Với c̄_p không đổi như Farokhi Ví dụ 7.5 (CH₄, 120 % không khí) thì ra bao nhiêu so với bảng?

In [8]:
import math, numpy as np, sympy as sp
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.interpolate import CubicSpline
from iapws import IAPWS97
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A5 = np.array([[1.0, 6.97], [1.5, 13.02], [2.0, 17.5], [2.5, 21.08], [3.0, 24.08], [4.0, 28.96], [5.0, 32.87], [7.5, 40.29], [10.0, 45.81], [15.0, 53.97], [20.0, 60.06], [25.0, 64.96], [30.0, 69.09], [40.0, 75.86], [50.0, 81.32], [75.0, 91.76], [100.0, 99.61], [101.325, 99.97], [125.0, 105.97], [150.0, 111.35], [175.0, 116.04], [200.0, 120.21]])                      # A-5: Psat [kPa], Tsat [C]  (P <= 200 kPa)
A4 = np.array([[5.0, 0.8725], [10.0, 1.2281], [15.0, 1.7057], [20.0, 2.3392], [25.0, 3.1698], [30.0, 4.2469], [35.0, 5.6291], [40.0, 7.3851], [45.0, 9.5953], [50.0, 12.352], [55.0, 15.763], [60.0, 19.947], [65.0, 25.043], [70.0, 31.202], [75.0, 38.597], [80.0, 47.416], [85.0, 57.868], [90.0, 70.183], [95.0, 84.609], [100.0, 101.42]])                      # A-4: T [C], Psat [kPa]     (5 C steps)
F26 = {"C(s)": 0.0, "H2(g)": 0.0, "N2(g)": 0.0, "O2(g)": 0.0, "CO(g)": -110530.0, "CO2(g)": -393520.0, "H2O(g)": -241820.0, "H2O(\u1409)": -285830.0, "H2O2(g)": -136310.0, "NH3(g)": -46190.0, "CH4(g)": -74850.0, "C2H2(g)": 226730.0, "C2H4(g)": 52280.0, "C2H6(g)": -84680.0, "C3H6(g)": 20410.0, "C3H8(g)": -103850.0, "C4H10(g)": -126150.0, "C8H18(g)": -208450.0, "C8H18(\u1409)": -249950.0, "C12H26(g)": -291010.0, "C6H6(g)": 82930.0, "CH3OH(g)": -200670.0, "CH3OH(\u1409)": -238660.0, "C2H5OH(g)": -235310.0, "C2H5OH(\u1409)": -277690.0, "O(g)": 249190.0, "H(g)": 218000.0, "N(g)": 472650.0, "OH(g)": 39460.0}                               # A-26: enthalpy of formation h_f0 [kJ/kmol] at 25 C, 1 atm
A27 = {"H2": {"M": 2.016, "HHV": 141800.0, "LHV": 120000.0}, "CH4": {"M": 16.043, "HHV": 55530.0, "LHV": 50050.0}, "C2H6": {"M": 30.07, "HHV": 51900.0, "LHV": 47520.0}, "C3H8": {"M": 44.097, "HHV": 50330.0, "LHV": 46340.0}, "C8H18": {"M": 114.231, "HHV": 47890.0, "LHV": 44430.0}}                               # A-27: M [kg/kmol], HHV, LHV [kJ/kg]
A2C = {"N2": [28.9, -0.001571, 8.081e-06, -2.873e-09, 1800], "O2": [25.48, 0.0152, -7.155e-06, 1.312e-09, 1800], "H2": [29.11, -0.001916, 4.003e-06, -8.704e-10, 1800], "CO": [28.16, 0.001675, 5.372e-06, -2.222e-09, 1800], "CO2": [22.26, 0.05981, -3.501e-05, 7.469e-09, 1800], "H2O": [32.24, 0.001923, 1.055e-05, -3.595e-09, 1800]}                               # A-2c: cp = a + bT + cT^2 + dT^3 [kJ/(kmol K)] and Tmax
HT = {"N2": [[270.0, 7849.0, 188.614], [280.0, 8141.0, 189.673], [290.0, 8432.0, 190.695], [298.0, 8669.0, 191.502], [300.0, 8723.0, 191.682], [310.0, 9014.0, 192.638], [320.0, 9306.0, 193.562], [330.0, 9597.0, 194.459], [340.0, 9888.0, 195.328], [350.0, 10180.0, 196.173], [360.0, 10471.0, 196.995], [370.0, 10763.0, 197.794], [380.0, 11055.0, 198.572], [390.0, 11347.0, 199.331], [400.0, 11640.0, 200.071], [410.0, 11932.0, 200.794], [420.0, 12225.0, 201.499], [430.0, 12518.0, 202.189], [440.0, 12811.0, 202.863], [450.0, 13105.0, 203.523], [460.0, 13399.0, 204.17], [470.0, 13693.0, 204.803], [480.0, 13988.0, 205.424], [490.0, 14285.0, 206.033], [500.0, 14581.0, 206.63], [510.0, 14876.0, 207.216], [520.0, 15172.0, 207.792], [530.0, 15469.0, 208.358], [540.0, 15766.0, 208.914], [550.0, 16064.0, 209.461], [560.0, 16363.0, 209.999], [570.0, 16662.0, 210.528], [580.0, 16962.0, 211.049], [590.0, 17262.0, 211.562], [600.0, 17563.0, 212.066], [610.0, 17864.0, 212.564], [620.0, 18166.0, 213.055], [630.0, 18468.0, 213.541], [640.0, 18772.0, 214.018], [650.0, 19075.0, 214.489], [660.0, 19380.0, 214.954], [670.0, 19685.0, 215.413], [680.0, 19991.0, 215.866], [690.0, 20297.0, 216.314], [700.0, 20604.0, 216.756], [710.0, 20912.0, 217.192], [720.0, 21220.0, 217.624], [730.0, 21529.0, 218.059], [740.0, 21839.0, 218.472], [750.0, 22149.0, 218.889], [760.0, 22460.0, 219.301], [770.0, 22772.0, 219.709], [780.0, 23085.0, 220.113], [790.0, 23398.0, 220.512], [800.0, 23714.0, 220.907], [810.0, 24027.0, 221.298], [820.0, 24342.0, 221.684], [830.0, 24658.0, 222.067], [840.0, 24974.0, 222.447], [850.0, 25292.0, 222.822], [860.0, 25610.0, 223.194], [870.0, 25928.0, 223.562], [880.0, 26248.0, 223.927], [890.0, 26568.0, 224.288], [900.0, 26890.0, 224.647], [910.0, 27210.0, 225.002], [920.0, 27532.0, 225.353], [930.0, 27854.0, 225.701], [940.0, 28178.0, 226.047], [950.0, 28501.0, 226.389], [960.0, 28826.0, 226.728], [970.0, 29151.0, 227.064], [980.0, 29476.0, 227.398], [990.0, 29803.0, 227.728], [1000.0, 30129.0, 228.057], [1020.0, 30784.0, 228.706], [1040.0, 31442.0, 229.344], [1060.0, 32101.0, 229.973], [1080.0, 32762.0, 230.591], [1100.0, 33426.0, 231.199], [1120.0, 34092.0, 231.799], [1140.0, 34760.0, 232.391], [1160.0, 35430.0, 232.973], [1180.0, 36104.0, 233.549], [1200.0, 36777.0, 234.115], [1220.0, 37452.0, 234.673], [1240.0, 38129.0, 235.223], [1260.0, 38807.0, 235.766], [1280.0, 39488.0, 236.302], [1300.0, 40170.0, 236.831], [1320.0, 40853.0, 237.353], [1340.0, 41539.0, 237.867], [1360.0, 42227.0, 238.376], [1380.0, 42915.0, 238.878], [1400.0, 43605.0, 239.375], [1420.0, 44295.0, 239.865], [1440.0, 44988.0, 240.35], [1460.0, 45682.0, 240.827], [1480.0, 46377.0, 241.301], [1500.0, 47073.0, 241.768], [1520.0, 47771.0, 242.228], [1540.0, 48470.0, 242.685], [1560.0, 49168.0, 243.137], [1580.0, 49869.0, 243.585], [1600.0, 50571.0, 244.028], [1620.0, 51275.0, 244.464], [1640.0, 51980.0, 244.896], [1660.0, 52686.0, 245.324], [1680.0, 53393.0, 245.747], [1700.0, 54099.0, 246.166], [1720.0, 54807.0, 246.58], [1740.0, 55516.0, 246.99], [1760.0, 56227.0, 247.396], [1780.0, 56938.0, 247.798], [1800.0, 57651.0, 248.195], [1820.0, 58363.0, 248.589], [1840.0, 59075.0, 248.979], [1860.0, 59790.0, 249.365], [1880.0, 60504.0, 249.748], [1900.0, 61220.0, 250.128], [1920.0, 61936.0, 250.502], [1940.0, 62654.0, 250.874], [1960.0, 63381.0, 251.242], [1980.0, 64090.0, 251.607], [2000.0, 64810.0, 251.969], [2050.0, 66612.0, 252.858], [2100.0, 68417.0, 253.726], [2150.0, 70226.0, 254.578], [2200.0, 72040.0, 255.412], [2250.0, 73856.0, 256.227], [2300.0, 75676.0, 257.027], [2350.0, 77496.0, 257.81], [2400.0, 79320.0, 258.58], [2450.0, 81149.0, 259.332], [2500.0, 82981.0, 260.073], [2550.0, 84814.0, 260.799], [2600.0, 86650.0, 261.512]], "O2": [[270.0, 7858.0, 202.128], [280.0, 8150.0, 203.191], [290.0, 8443.0, 204.218], [298.0, 8682.0, 205.033], [300.0, 8736.0, 205.213], [310.0, 9030.0, 206.177], [320.0, 9325.0, 207.112], [330.0, 9620.0, 208.02], [340.0, 9916.0, 208.904], [350.0, 10213.0, 209.765], [360.0, 10511.0, 210.604], [370.0, 10809.0, 211.423], [380.0, 11109.0, 212.222], [390.0, 11409.0, 213.002], [400.0, 11711.0, 213.765], [410.0, 12012.0, 214.51], [420.0, 12314.0, 215.241], [430.0, 12618.0, 215.955], [440.0, 12923.0, 216.656], [450.0, 13228.0, 217.342], [460.0, 13525.0, 218.016], [470.0, 13842.0, 218.676], [480.0, 14151.0, 219.326], [490.0, 14460.0, 219.963], [500.0, 14770.0, 220.589], [510.0, 15082.0, 221.206], [520.0, 15395.0, 221.812], [530.0, 15708.0, 222.409], [540.0, 16022.0, 222.997], [550.0, 16338.0, 223.576], [560.0, 16654.0, 224.146], [570.0, 16971.0, 224.708], [580.0, 17290.0, 225.262], [590.0, 17609.0, 225.808], [600.0, 17929.0, 226.346], [610.0, 18250.0, 226.877], [620.0, 18572.0, 227.4], [630.0, 18895.0, 227.918], [640.0, 19219.0, 228.429], [650.0, 19544.0, 228.932], [660.0, 19870.0, 229.43], [670.0, 20197.0, 229.92], [680.0, 20524.0, 230.405], [690.0, 20854.0, 230.885], [700.0, 21184.0, 231.358], [710.0, 21514.0, 231.827], [720.0, 21845.0, 232.291], [730.0, 22177.0, 232.748], [740.0, 22510.0, 233.201], [750.0, 22844.0, 233.649], [760.0, 23178.0, 234.091], [770.0, 23513.0, 234.528], [780.0, 23850.0, 234.96], [790.0, 24186.0, 235.387], [800.0, 24523.0, 235.81], [810.0, 24861.0, 236.23], [820.0, 25199.0, 236.644], [830.0, 25537.0, 237.055], [840.0, 25877.0, 237.462], [850.0, 26218.0, 237.864], [860.0, 26559.0, 238.264], [870.0, 26899.0, 238.66], [880.0, 27242.0, 239.051], [890.0, 27584.0, 239.439], [900.0, 27928.0, 239.823], [910.0, 28272.0, 240.203], [920.0, 28616.0, 240.58], [930.0, 28960.0, 240.953], [940.0, 29306.0, 241.323], [950.0, 29652.0, 241.689], [960.0, 29999.0, 242.052], [970.0, 30345.0, 242.411], [980.0, 30692.0, 242.768], [990.0, 31041.0, 243.122], [1000.0, 31389.0, 243.471], [1020.0, 32088.0, 244.164], [1040.0, 32789.0, 244.844], [1060.0, 33490.0, 245.513], [1080.0, 34194.0, 246.171], [1100.0, 34899.0, 246.818], [1120.0, 35606.0, 247.454], [1140.0, 36314.0, 248.081], [1160.0, 37023.0, 248.698], [1180.0, 37734.0, 249.307], [1200.0, 38447.0, 249.906], [1220.0, 39162.0, 250.497], [1240.0, 39877.0, 251.079], [1260.0, 40594.0, 251.653], [1280.0, 41312.0, 252.219], [1300.0, 42033.0, 252.776], [1320.0, 42753.0, 253.325], [1340.0, 43475.0, 253.868], [1360.0, 44198.0, 254.404], [1380.0, 44923.0, 254.932], [1400.0, 45648.0, 255.454], [1420.0, 46374.0, 255.968], [1440.0, 47102.0, 256.475], [1460.0, 47831.0, 256.978], [1480.0, 48561.0, 257.474], [1500.0, 49292.0, 257.965], [1520.0, 50024.0, 258.45], [1540.0, 50756.0, 258.928], [1560.0, 51490.0, 259.402], [1580.0, 52224.0, 259.87], [1600.0, 52961.0, 260.333], [1620.0, 53696.0, 260.791], [1640.0, 54434.0, 261.242], [1660.0, 55172.0, 261.69], [1680.0, 55912.0, 262.132], [1700.0, 56652.0, 262.571], [1720.0, 57394.0, 263.005], [1740.0, 58136.0, 263.435], [1760.0, 58880.0, 263.861], [1780.0, 59624.0, 264.283], [1800.0, 60371.0, 264.701], [1820.0, 61118.0, 265.113], [1840.0, 61866.0, 265.521], [1860.0, 62616.0, 265.925], [1880.0, 63365.0, 266.326], [1900.0, 64116.0, 266.722], [1920.0, 64868.0, 267.115], [1940.0, 65620.0, 267.505], [1960.0, 66374.0, 267.891], [1980.0, 67127.0, 268.275], [2000.0, 67881.0, 268.655], [2050.0, 69772.0, 269.588], [2100.0, 71668.0, 270.504], [2150.0, 73573.0, 271.399], [2200.0, 75484.0, 272.278], [2250.0, 77397.0, 273.136], [2300.0, 79316.0, 273.98], [2350.0, 81243.0, 274.809], [2400.0, 83174.0, 275.625], [2450.0, 85112.0, 276.424], [2500.0, 87057.0, 277.207], [2550.0, 89004.0, 277.979], [2600.0, 90956.0, 278.738]], "CO2": [[270.0, 8335.0, 210.062], [280.0, 8697.0, 211.376], [290.0, 9063.0, 212.66], [298.0, 9364.0, 213.685], [300.0, 9431.0, 213.915], [310.0, 9807.0, 215.146], [320.0, 10186.0, 216.351], [330.0, 10570.0, 217.534], [340.0, 10959.0, 218.694], [350.0, 11351.0, 219.831], [360.0, 11748.0, 220.948], [370.0, 12148.0, 222.044], [380.0, 12552.0, 223.122], [390.0, 12960.0, 224.182], [400.0, 13372.0, 225.225], [410.0, 13787.0, 226.25], [420.0, 14206.0, 227.258], [430.0, 14628.0, 228.252], [440.0, 15054.0, 229.23], [450.0, 15483.0, 230.194], [460.0, 15916.0, 231.144], [470.0, 16351.0, 232.08], [480.0, 16791.0, 233.004], [490.0, 17232.0, 233.916], [500.0, 17678.0, 234.814], [510.0, 18126.0, 235.7], [520.0, 18576.0, 236.575], [530.0, 19029.0, 237.439], [540.0, 19485.0, 238.292], [550.0, 19945.0, 239.135], [560.0, 20407.0, 239.962], [570.0, 20870.0, 240.789], [580.0, 21337.0, 241.602], [590.0, 21807.0, 242.405], [600.0, 22280.0, 243.199], [610.0, 22754.0, 243.983], [620.0, 23231.0, 244.758], [630.0, 23709.0, 245.524], [640.0, 24190.0, 246.282], [650.0, 24674.0, 247.032], [660.0, 25160.0, 247.773], [670.0, 25648.0, 248.507], [680.0, 26138.0, 249.233], [690.0, 26631.0, 249.952], [700.0, 27125.0, 250.663], [710.0, 27622.0, 251.368], [720.0, 28121.0, 252.065], [730.0, 28622.0, 252.755], [740.0, 29124.0, 253.439], [750.0, 29629.0, 254.117], [760.0, 30135.0, 254.787], [770.0, 30644.0, 255.452], [780.0, 31154.0, 256.11], [790.0, 31665.0, 256.762], [800.0, 32179.0, 257.408], [810.0, 32694.0, 258.048], [820.0, 33212.0, 258.682], [830.0, 33730.0, 259.311], [840.0, 34251.0, 259.934], [850.0, 34773.0, 260.551], [860.0, 35296.0, 261.164], [870.0, 35821.0, 261.77], [880.0, 36347.0, 262.371], [890.0, 36876.0, 262.968], [900.0, 37405.0, 263.559], [910.0, 37935.0, 264.146], [920.0, 38467.0, 264.728], [930.0, 39000.0, 265.304], [940.0, 39535.0, 265.877], [950.0, 40070.0, 266.444], [960.0, 40607.0, 267.007], [970.0, 41145.0, 267.566], [980.0, 41685.0, 268.119], [990.0, 42226.0, 268.67], [1000.0, 42769.0, 269.215], [1020.0, 43859.0, 270.293], [1040.0, 44953.0, 271.354], [1060.0, 46051.0, 272.4], [1080.0, 47153.0, 273.43], [1100.0, 48258.0, 274.445], [1120.0, 49369.0, 275.444], [1140.0, 50484.0, 276.43], [1160.0, 51602.0, 277.403], [1180.0, 52724.0, 278.361], [1200.0, 53848.0, 279.306], [1220.0, 54977.0, 280.238], [1240.0, 56108.0, 281.158], [1260.0, 57244.0, 282.066], [1280.0, 58381.0, 282.962], [1300.0, 59522.0, 283.847], [1320.0, 60666.0, 284.722], [1340.0, 61813.0, 285.586], [1360.0, 62963.0, 286.439], [1380.0, 64116.0, 287.283], [1400.0, 65271.0, 288.106], [1420.0, 66427.0, 288.934], [1440.0, 67586.0, 289.743], [1460.0, 68748.0, 290.542], [1480.0, 66911.0, 291.333], [1500.0, 71078.0, 292.114], [1520.0, 72246.0, 292.888], [1540.0, 73417.0, 293.653], [1560.0, 74590.0, 294.411], [1580.0, 76767.0, 295.161], [1600.0, 76944.0, 295.901], [1620.0, 78123.0, 296.632], [1640.0, 79303.0, 297.356], [1660.0, 80486.0, 298.072], [1680.0, 81670.0, 298.781], [1700.0, 82856.0, 299.482], [1720.0, 84043.0, 300.177], [1740.0, 85231.0, 300.863], [1760.0, 86420.0, 301.543], [1780.0, 87612.0, 302.217], [1800.0, 88806.0, 302.884], [1820.0, 90000.0, 303.544], [1840.0, 91196.0, 304.198], [1860.0, 92394.0, 304.845], [1880.0, 93593.0, 305.487], [1900.0, 94793.0, 306.122], [1920.0, 95995.0, 306.751], [1940.0, 97197.0, 307.374], [1960.0, 98401.0, 307.992], [1980.0, 99606.0, 308.604], [2000.0, 100804.0, 309.21], [2050.0, 103835.0, 310.701], [2100.0, 106864.0, 312.16], [2150.0, 109898.0, 313.589], [2200.0, 112939.0, 314.988], [2250.0, 115984.0, 316.356], [2300.0, 119035.0, 317.695], [2350.0, 122091.0, 319.011], [2400.0, 125152.0, 320.302], [2450.0, 128219.0, 321.566], [2500.0, 131290.0, 322.808], [2550.0, 134368.0, 324.026], [2600.0, 137449.0, 325.222]], "CO": [[270.0, 7849.0, 194.654], [280.0, 8140.0, 195.713], [290.0, 8432.0, 196.735], [298.0, 8669.0, 197.543], [300.0, 8723.0, 197.723], [310.0, 9014.0, 198.678], [320.0, 9306.0, 199.603], [330.0, 9597.0, 200.5], [340.0, 9889.0, 201.371], [350.0, 10181.0, 202.217], [360.0, 10473.0, 203.04], [370.0, 10765.0, 203.842], [380.0, 11058.0, 204.622], [390.0, 11351.0, 205.383], [400.0, 11644.0, 206.125], [410.0, 11938.0, 206.85], [420.0, 12232.0, 207.549], [430.0, 12526.0, 208.252], [440.0, 12821.0, 208.929], [450.0, 13116.0, 209.593], [460.0, 13412.0, 210.243], [470.0, 13708.0, 210.88], [480.0, 14005.0, 211.504], [490.0, 14302.0, 212.117], [500.0, 14600.0, 212.719], [510.0, 14898.0, 213.31], [520.0, 15197.0, 213.89], [530.0, 15497.0, 214.46], [540.0, 15797.0, 215.02], [550.0, 16097.0, 215.572], [560.0, 16399.0, 216.115], [570.0, 16701.0, 216.649], [580.0, 17003.0, 217.175], [590.0, 17307.0, 217.693], [600.0, 17611.0, 218.204], [610.0, 17915.0, 218.708], [620.0, 18221.0, 219.205], [630.0, 18527.0, 219.695], [640.0, 18833.0, 220.179], [650.0, 19141.0, 220.656], [660.0, 19449.0, 221.127], [670.0, 19758.0, 221.592], [680.0, 20068.0, 222.052], [690.0, 20378.0, 222.505], [700.0, 20690.0, 222.953], [710.0, 21002.0, 223.396], [720.0, 21315.0, 223.833], [730.0, 21628.0, 224.265], [740.0, 21943.0, 224.692], [750.0, 22258.0, 225.115], [760.0, 22573.0, 225.533], [770.0, 22890.0, 225.947], [780.0, 23208.0, 226.357], [790.0, 23526.0, 226.762], [800.0, 23844.0, 227.162], [810.0, 24164.0, 227.559], [820.0, 24483.0, 227.952], [830.0, 24803.0, 228.339], [840.0, 25124.0, 228.724], [850.0, 25446.0, 229.106], [860.0, 25768.0, 229.482], [870.0, 26091.0, 229.856], [880.0, 26415.0, 230.227], [890.0, 26740.0, 230.593], [900.0, 27066.0, 230.957], [910.0, 27392.0, 231.317], [920.0, 27719.0, 231.674], [930.0, 28046.0, 232.028], [940.0, 28375.0, 232.379], [950.0, 28703.0, 232.727], [960.0, 29033.0, 233.072], [970.0, 29362.0, 233.413], [980.0, 29693.0, 233.752], [990.0, 30024.0, 234.088], [1000.0, 30355.0, 234.421], [1020.0, 31020.0, 235.079], [1040.0, 31688.0, 235.728], [1060.0, 32357.0, 236.364], [1080.0, 33029.0, 236.992], [1100.0, 33702.0, 237.609], [1120.0, 34377.0, 238.217], [1140.0, 35054.0, 238.817], [1160.0, 35733.0, 239.407], [1180.0, 36406.0, 239.989], [1200.0, 37095.0, 240.562], [1220.0, 37780.0, 241.128], [1240.0, 38466.0, 241.686], [1260.0, 39154.0, 242.236], [1280.0, 39844.0, 242.78], [1300.0, 40534.0, 243.316], [1320.0, 41226.0, 243.844], [1340.0, 41919.0, 244.366], [1360.0, 42613.0, 244.88], [1380.0, 43309.0, 245.388], [1400.0, 44007.0, 245.889], [1420.0, 44707.0, 246.385], [1440.0, 45408.0, 246.876], [1460.0, 46110.0, 247.36], [1480.0, 46813.0, 247.839], [1500.0, 47517.0, 248.312], [1520.0, 48222.0, 248.778], [1540.0, 48928.0, 249.24], [1560.0, 49635.0, 249.695], [1580.0, 50344.0, 250.147], [1600.0, 51053.0, 250.592], [1620.0, 51763.0, 251.033], [1640.0, 52472.0, 251.47], [1660.0, 53184.0, 251.901], [1680.0, 53895.0, 252.329], [1700.0, 54609.0, 252.751], [1720.0, 55323.0, 253.169], [1740.0, 56039.0, 253.582], [1760.0, 56756.0, 253.991], [1780.0, 57473.0, 254.398], [1800.0, 58191.0, 254.797], [1820.0, 58910.0, 255.194], [1840.0, 59629.0, 255.587], [1860.0, 60351.0, 255.976], [1880.0, 61072.0, 256.361], [1900.0, 61794.0, 256.743], [1920.0, 62516.0, 257.122], [1940.0, 63238.0, 257.497], [1960.0, 63961.0, 257.868], [1980.0, 64684.0, 258.236], [2000.0, 65408.0, 258.6], [2050.0, 67224.0, 259.494], [2100.0, 69044.0, 260.37], [2150.0, 70864.0, 261.226], [2200.0, 72688.0, 262.065], [2250.0, 74516.0, 262.887], [2300.0, 76345.0, 263.692], [2350.0, 78178.0, 264.48], [2400.0, 80015.0, 265.253], [2450.0, 81852.0, 266.012], [2500.0, 83692.0, 266.755], [2550.0, 85537.0, 267.485], [2600.0, 87383.0, 268.202]], "H2": [[270.0, 7657.0, 127.719], [280.0, 7945.0, 128.765], [290.0, 8233.0, 129.775], [298.0, 8468.0, 130.574], [300.0, 8522.0, 130.754], [320.0, 9100.0, 132.621], [340.0, 9680.0, 134.378], [360.0, 10262.0, 136.039], [380.0, 10843.0, 137.612], [400.0, 11426.0, 139.106], [420.0, 12010.0, 140.529], [440.0, 12594.0, 141.888], [460.0, 13179.0, 143.187], [480.0, 13764.0, 144.432], [500.0, 14350.0, 145.628], [520.0, 14935.0, 146.775], [560.0, 16107.0, 148.945], [600.0, 17280.0, 150.968], [640.0, 18453.0, 152.863], [680.0, 19630.0, 154.645], [720.0, 20807.0, 156.328], [760.0, 21988.0, 157.923], [800.0, 23171.0, 159.44], [840.0, 24359.0, 160.891], [880.0, 25551.0, 162.277], [920.0, 26747.0, 163.607], [960.0, 27948.0, 164.884], [1000.0, 29154.0, 166.114], [1040.0, 30364.0, 167.3], [1080.0, 31580.0, 168.449], [1120.0, 32802.0, 169.56], [1160.0, 34028.0, 170.636], [1200.0, 35262.0, 171.682], [1240.0, 36502.0, 172.698], [1280.0, 37749.0, 173.687], [1320.0, 39002.0, 174.652], [1360.0, 40263.0, 175.593], [1400.0, 41530.0, 176.51], [1440.0, 42808.0, 177.41], [1480.0, 44091.0, 178.291], [1520.0, 45384.0, 179.153], [1560.0, 46683.0, 179.995], [1600.0, 47990.0, 180.82], [1640.0, 49303.0, 181.632], [1680.0, 50622.0, 182.428], [1720.0, 51947.0, 183.208], [1760.0, 53279.0, 183.973], [1800.0, 54618.0, 184.724], [1840.0, 55962.0, 185.463], [1880.0, 57311.0, 186.19], [1920.0, 58668.0, 186.904], [1960.0, 60031.0, 187.607], [2000.0, 61400.0, 188.297], [2050.0, 63119.0, 189.148], [2100.0, 64847.0, 189.979], [2150.0, 66584.0, 190.796], [2200.0, 68328.0, 191.598], [2250.0, 70080.0, 192.385], [2300.0, 71839.0, 193.159], [2350.0, 73608.0, 193.921], [2400.0, 75383.0, 194.669], [2450.0, 77168.0, 195.403], [2500.0, 78960.0, 196.125], [2550.0, 80755.0, 196.837], [2600.0, 82558.0, 197.539]], "H2O": [[270.0, 8961.0, 185.399], [280.0, 9296.0, 186.616], [290.0, 9631.0, 187.791], [298.0, 9904.0, 188.72], [300.0, 9966.0, 188.928], [310.0, 10302.0, 190.03], [320.0, 10639.0, 191.098], [330.0, 10976.0, 192.136], [340.0, 11314.0, 193.144], [350.0, 11652.0, 194.125], [360.0, 11992.0, 195.081], [370.0, 12331.0, 196.012], [380.0, 12672.0, 196.92], [390.0, 13014.0, 197.807], [400.0, 13356.0, 198.673], [410.0, 13699.0, 199.521], [420.0, 14043.0, 200.35], [430.0, 14388.0, 201.16], [440.0, 14734.0, 201.955], [450.0, 15080.0, 202.734], [460.0, 15428.0, 203.497], [470.0, 15777.0, 204.247], [480.0, 16126.0, 204.982], [490.0, 16477.0, 205.705], [500.0, 16828.0, 206.413], [510.0, 17181.0, 207.112], [520.0, 17534.0, 207.799], [530.0, 17889.0, 208.475], [540.0, 18245.0, 209.139], [550.0, 18601.0, 209.795], [560.0, 18959.0, 210.44], [570.0, 19318.0, 211.075], [580.0, 19678.0, 211.702], [590.0, 20039.0, 212.32], [600.0, 20402.0, 212.92], [610.0, 20765.0, 213.529], [620.0, 21130.0, 214.122], [630.0, 21495.0, 214.707], [640.0, 21862.0, 215.285], [650.0, 22230.0, 215.856], [660.0, 22600.0, 216.419], [670.0, 22970.0, 216.976], [680.0, 23342.0, 217.527], [690.0, 23714.0, 218.071], [700.0, 24088.0, 218.61], [710.0, 24464.0, 219.142], [720.0, 24840.0, 219.668], [730.0, 25218.0, 220.189], [740.0, 25597.0, 220.707], [750.0, 25977.0, 221.215], [760.0, 26358.0, 221.72], [770.0, 26741.0, 222.221], [780.0, 27125.0, 222.717], [790.0, 27510.0, 223.207], [800.0, 27896.0, 223.693], [810.0, 28284.0, 224.174], [820.0, 28672.0, 224.651], [830.0, 29062.0, 225.123], [840.0, 29454.0, 225.592], [850.0, 29846.0, 226.057], [860.0, 30240.0, 226.517], [870.0, 30635.0, 226.973], [880.0, 31032.0, 227.426], [890.0, 31429.0, 227.875], [900.0, 31828.0, 228.321], [910.0, 32228.0, 228.763], [920.0, 32629.0, 229.202], [930.0, 33032.0, 229.637], [940.0, 33436.0, 230.07], [950.0, 33841.0, 230.499], [960.0, 34247.0, 230.924], [970.0, 34653.0, 231.347], [980.0, 35061.0, 231.767], [990.0, 35472.0, 232.184], [1000.0, 35882.0, 232.597], [1020.0, 36709.0, 233.415], [1040.0, 37542.0, 234.223], [1060.0, 38380.0, 235.02], [1080.0, 39223.0, 235.806], [1100.0, 40071.0, 236.584], [1120.0, 40923.0, 237.352], [1140.0, 41780.0, 238.11], [1160.0, 42642.0, 238.859], [1180.0, 43509.0, 239.6], [1200.0, 44380.0, 240.333], [1220.0, 45256.0, 241.057], [1240.0, 46137.0, 241.773], [1260.0, 47022.0, 242.482], [1280.0, 47912.0, 243.183], [1300.0, 48807.0, 243.877], [1320.0, 49707.0, 244.564], [1340.0, 50612.0, 245.243], [1360.0, 51521.0, 245.915], [1380.0, 52434.0, 246.582], [1400.0, 53351.0, 247.241], [1420.0, 54273.0, 247.895], [1440.0, 55198.0, 248.543], [1460.0, 56128.0, 249.185], [1480.0, 57062.0, 249.82], [1500.0, 57999.0, 250.45], [1520.0, 58942.0, 251.074], [1540.0, 59888.0, 251.693], [1560.0, 60838.0, 252.305], [1580.0, 61792.0, 252.912], [1600.0, 62748.0, 253.513], [1620.0, 63709.0, 254.111], [1640.0, 64675.0, 254.703], [1660.0, 65643.0, 255.29], [1680.0, 66614.0, 255.873], [1700.0, 67589.0, 256.45], [1720.0, 68567.0, 257.022], [1740.0, 69550.0, 257.589], [1760.0, 70535.0, 258.151], [1780.0, 71523.0, 258.708], [1800.0, 72513.0, 259.262], [1820.0, 73507.0, 259.811], [1840.0, 74506.0, 260.357], [1860.0, 75506.0, 260.898], [1880.0, 76511.0, 261.436], [1900.0, 77517.0, 261.969], [1920.0, 78527.0, 262.497], [1940.0, 79540.0, 263.022], [1960.0, 80555.0, 263.542], [1980.0, 81573.0, 264.059], [2000.0, 82593.0, 264.571], [2050.0, 85156.0, 265.838], [2100.0, 87735.0, 267.081], [2150.0, 90330.0, 268.301], [2200.0, 92940.0, 269.5], [2250.0, 95562.0, 270.679], [2300.0, 98199.0, 271.839], [2350.0, 100846.0, 272.978], [2400.0, 103508.0, 274.098], [2450.0, 106183.0, 275.201], [2500.0, 108868.0, 276.286], [2550.0, 111565.0, 277.354], [2600.0, 114273.0, 278.407]]}                                # A-18..A-23: rows [T, h, s0] (h in kJ/kmol, s0 in kJ/(kmol K)), 270-2600 K; s0 cells flagged in book_errata.json replaced by the expected value
M_AIR_ROUND, M_AIR_EXACT = 29.0, 28.97 # slides round 28.97 (A-1) to 29
def Tsat_A5(P):  return float(np.interp(P, A5[:, 0], A5[:, 1]))                 # the book's method: linear interpolation
def Tsat_iapws(P): return IAPWS97(P=P / 1000.0, x=0).T - 273.15                    # independent route: IF97
def hbar(g, T):  tab = np.array(HT[g]); return float(np.interp(T, tab[:, 0], tab[:, 1]))
def sens_table(g, T): return hbar(g, T) - hbar(g, 298.0)                           # h(T) - h(298 K), table column h
_SP = {g: CubicSpline([r[0] for r in HT[g]], [r[2] for r in HT[g]]) for g in HT}
def sens_Tds(g, T): return T * float(_SP[g](T)) - 298.0 * float(_SP[g](298.0)) - quad(lambda t: float(_SP[g](t)), 298.0, T)[0]   # integral of T ds0 (s0 column)
def sens_poly(g, T):
    a, b, c, d, Tmax = A2C[g]; assert T <= Tmax; return a * (T - 298.0) + b / 2 * (T**2 - 298.0**2) + c / 3 * (T**3 - 298.0**3) + d / 4 * (T**4 - 298.0**4)
def comp_el(formula):
    import re; e = {'C': 0, 'H': 0, 'O': 0, 'N': 0}
    for el, n in re.findall(r'([A-Z])(\d*)', formula): e[el] += int(n) if n else 1
    return e
def balance(formula, lam):
    """sympy: fuel + lam*a_th (O2 + 3.76 N2) -> nCO2 CO2 + nH2O H2O + nO2 O2 + nN2 N2 (exact rationals); returns kmol per kmol of fuel."""
    e = comp_el(formula); a, nc, nh, nn, no2 = sp.symbols('a nc nh nn no2'); r = sp.Rational(376, 100); lam = sp.nsimplify(lam)
    ath = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * a, 2 * nc + nh), sp.Eq(e['N'] + 2 * r * a, 2 * nn)], [a, nc, nh, nn], dict=True)[0][a]
    s = sp.solve([sp.Eq(e['C'], nc), sp.Eq(e['H'], 2 * nh), sp.Eq(e['O'] + 2 * lam * ath, 2 * nc + nh + 2 * no2), sp.Eq(e['N'] + 2 * r * lam * ath, 2 * nn)], [nc, nh, no2, nn], dict=True)[0]
    return dict(ath=float(ath), CO2=float(s[nc]), H2O=float(s[nh]), O2=float(s[no2]), N2=float(s[nn]), air=float(lam * ath * (1 + r)))
hf = F26
def Tad(react, prod, sens):
    Hr = sum(n * (hf[k] + (sens(g, Tr) if g else 0.0)) for g, n, k, Tr in react)
    return brentq(lambda T: sum(n * (hf[k] + (sens(g, T) if g else 0.0)) for g, n, k in prod) - Hr, 300.0, 3200.0, xtol=1e-6)
# 15-70: H2 at 7 C (280 K) with 20 % excess air at 280 K -> H2O + 0.1 O2 + 2.256 N2
r70 = [('H2', 1.0, 'H2(g)', 280.0), ('O2', 0.6, 'O2(g)', 280.0), ('N2', 0.6 * 3.76, 'N2(g)', 280.0)]; p70 = [('H2O', 1.0, 'H2O(g)'), ('O2', 0.1, 'O2(g)'), ('N2', 0.6 * 3.76, 'N2(g)')]
# Example 15-10: CH4 + 3 (O2 + 3.76 N2) -> CO2 + 2 H2O + O2 + 11.28 N2 at 25 C
r10 = [(None, 1.0, 'CH4(g)', 298.0), ('O2', 3.0, 'O2(g)', 298.0), ('N2', 11.28, 'N2(g)', 298.0)]; p10 = [('CO2', 1.0, 'CO2(g)'), ('H2O', 2.0, 'H2O(g)'), ('O2', 1.0, 'O2(g)'), ('N2', 11.28, 'N2(g)')]
# Farokhi Example 7.5: CH4 + 2.4 (O2 + 3.76 N2) -> CO2 + 2 H2O + 0.4 O2 + 9.02 N2 at 25 C; constant cp 61.9, 52.3, 37.8, 33.6 kJ/(kmol K) and h_f(CH4) = -74,873
r75 = [(None, 1.0, 'CH4(g)', 298.0), ('O2', 2.4, 'O2(g)', 298.0), ('N2', 2.4 * 3.76, 'N2(g)', 298.0)]; p75 = [('CO2', 1.0, 'CO2(g)'), ('H2O', 2.0, 'H2O(g)'), ('O2', 0.4, 'O2(g)'), ('N2', 2.4 * 3.76, 'N2(g)')]
for tag, rr, pp, printed in (('Cengel 15-70', r70, p70, '2251.4'), ('Cengel Ex 15-10', r10, p10, '1789'), ('Farokhi Ex 7.5 (tables)', r75, p75, '1953 with constant cp')):
    print(f"{tag:24s}: table h {Tad(rr, pp, sens_table):7.1f} K | integral of T ds0 {Tad(rr, pp, sens_Tds):7.1f} K | printed {printed}")
cpsum = 61.9 + 2 * 52.3 + 0.4 * 37.8 + 9.02 * 33.6; dH = 393522 + 2 * 241827 - 74873
print(f"Farokhi constant-cp route: sum n cp = {cpsum:.1f}; T = 298 + {dH:,.0f}/{cpsum:.1f} = {298 + dH / cpsum:.0f} K")

Cengel 15-70            : table h  2251.4 K | integral of T ds0  2251.5 K | printed 2251.4


Cengel Ex 15-10         : table h  1789.0 K | integral of T ds0  1789.0 K | printed 1789


Farokhi Ex 7.5 (tables) : table h  2069.9 K | integral of T ds0  2069.9 K | printed 1953 with constant cp
Farokhi constant-cp route: sum n cp = 484.7; T = 298 + 802,303/484.7 = 1953 K


**Đọc kết quả:** 15–70: 2251,4 K (in 2251,4); Ví dụ 15–10: 1789 K (in 1789): cả hai khớp. Tích phân T ds° cho cùng nhiệt độ trong vài phần nghìn. Với CH₄, 120 % không khí, bảng cho 2070 K còn đường c̄_p không đổi của Farokhi cho 1953 K: c̄_p lấy gần 300 K đánh giá thấp nhiệt độ khoảng 117 K. Phân ly (slide tr.8) còn làm nhiệt độ thực thấp hơn nữa.